In [0]:
customers=spark.table("silver.customers")
orders=spark.table("silver.orders")
order_items=spark.table("silver.order_items")
payments=spark.table("silver.payments")
products=spark.table("silver.products")
sellers=spark.table("silver.sellers")
reviews=spark.table("silver.reviews")

**CUSTOMER DIMENSION**

In [0]:
dim_customers=customers.select("customer_id","customer_unique_id","customer_city","customer_state").dropDuplicates()

In [0]:
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_customers")

**PRODUCT DIMENSION**

In [0]:
dim_products=products.select("product_id","product_category_name","product_weight_g").dropDuplicates()

In [0]:
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_products")


**SELLER DIMENSION**

In [0]:
dim_sellers=sellers.select("seller_id","seller_city","seller_state").dropDuplicates()

In [0]:
dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_sellers")


**DATE DIMENSION**

In [0]:
from pyspark.sql.functions import *

dim_date=orders.select(col("order_purchase_timestamp").alias("date")).distinct()

In [0]:
dim_date=dim_date \
    .withColumn("year",year("date")) \
    .withColumn("month",month("date")) \
    .withColumn("month_name",date_format("date","MMMM")) \
    .withColumn("quarter",quarter("date")) 
    

In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_date")

**FACT ORDERS TABLE**

In [0]:
fact_orders=orders.join(order_items,"order_id","inner")

In [0]:
fact_orders=fact_orders.join(payments,"order_id","left")

In [0]:
fact_orders=fact_orders.join(reviews,"order_id","left")

In [0]:
fact_orders=fact_orders.select(
    "order_id","customer_id","product_id","review_score","order_purchase_timestamp","payment_value","seller_id"
)

In [0]:
fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_orders")

**SALES SUMMARY TABLE**

In [0]:
from pyspark.sql.functions import sum

sales_summary=fact_orders.groupBy("seller_id").agg(sum("payment_value").alias("total_sales"))

In [0]:
sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.sales_summary")


**CUSTOMER SUMMARY**

In [0]:
customers_summary=fact_orders.groupBy("customer_id").agg(sum("payment_value").alias("customer_spend"))

In [0]:
customers_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.customers_summary")


**PRODUCTS SUMMARY**

In [0]:
from pyspark.sql.functions import count
product_summary=fact_orders.groupBy("product_id").agg(count("*").alias("orders_count"))

In [0]:
product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.product_summary")

In [0]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|gold    |customers_summary|false      |
|gold    |dim_customers    |false      |
|gold    |dim_date         |false      |
|gold    |dim_products     |false      |
|gold    |dim_sellers      |false      |
|gold    |fact_orders      |false      |
|gold    |product_summary  |false      |
|gold    |sales_summary    |false      |
+--------+-----------------+-----------+

